# Audio Representations & Spectrograms

Companion notebook for the [Audio Representations lesson](https://ml-viz.vercel.app/courses/speech-audio/01-audio-representations).

We synthesize a waveform, compute its **spectrogram** with a from-scratch **STFT**, and apply a
**mel filterbank** — seeing how a 1-D wave becomes a 2-D time-frequency image and why the window
length sets a time-frequency trade-off. Pure NumPy + Matplotlib (no audio libraries).

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 1.5,
})

## 1 — A waveform with changing frequency

A 'chirp' that sweeps from low to high pitch — so the spectrogram will show a clear diagonal,
demonstrating that we recover *when* each frequency occurs.

In [ ]:
fs = 8000                    # sampling rate (Hz) -> Nyquist limit 4000 Hz
t = np.linspace(0, 2, 2*fs, endpoint=False)
freq = 200 + 600 * t / 2     # frequency sweeps 200 -> 800 Hz
wave = np.sin(2*np.pi*np.cumsum(freq)/fs)
print(f'{len(wave)} samples for {t[-1]+1/fs:.0f}s at {fs} Hz (Nyquist = {fs//2} Hz)')

## 2 — STFT: slide a window, take the Fourier transform of each

The spectrogram is |STFT|: rows = frequency, columns = time. The diagonal ridge is the rising pitch
— time-frequency structure a single FFT would have lost.

In [ ]:
def stft(x, win, hop):
    window = np.hanning(win)
    frames = [np.abs(np.fft.rfft(x[i:i+win] * window))
              for i in range(0, len(x)-win, hop)]
    return np.array(frames).T          # (freq_bins, time_frames)

spec = stft(wave, win=256, hop=64)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.imshow(np.log(spec+1e-6), origin='lower', aspect='auto', cmap='magma',
          extent=[0, 2, 0, fs/2])
ax.set_xlabel('time (s)'); ax.set_ylabel('frequency (Hz)')
ax.set_title('Spectrogram (STFT): the rising-pitch chirp is a diagonal ridge')
plt.tight_layout(); plt.show()

## 3 — The time-frequency resolution trade-off

A short window pins down time but smears frequency; a long window does the reverse. Same signal,
different windows.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, win in zip(axes, [64, 1024]):
    s = stft(wave, win=win, hop=win//4)
    ax.imshow(np.log(s+1e-6), origin='lower', aspect='auto', cmap='magma', extent=[0,2,0,fs/2])
    ax.set_title(f'window={win} ({"sharp time" if win==64 else "sharp frequency"})')
    ax.set_xlabel('time (s)')
axes[0].set_ylabel('frequency (Hz)')
plt.tight_layout(); plt.show()
print('Short window -> crisp in time, blurry in frequency. Long window -> the opposite.')

## ✏️ Your turn

**Exercise.** Implement `nyquist(fs)` (the highest faithfully representable frequency) and
`n_frames(signal_len, win, hop)` (how many STFT windows fit in a signal). These two govern the shape
of every spectrogram.

In [ ]:
def nyquist(fs):
    # TODO(you): the Nyquist frequency for sampling rate fs
    return ...

def n_frames(signal_len, win, hop):
    # TODO(you): number of windows of size `win` stepping by `hop` that fit (matching the stft loop)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert nyquist(16000) == 8000
assert nyquist(44100) == 22050
# matches the actual number of columns the stft above produced
assert n_frames(len(wave), 256, 64) == spec.shape[1]
assert n_frames(1000, 100, 100) == 9
print('\u2713 Nyquist and frame-count are correct')

<details>
<summary>Solution</summary>

```python
def nyquist(fs):
    return fs // 2

def n_frames(signal_len, win, hop):
    return len(range(0, signal_len - win, hop))
```

Nyquist caps the frequencies you can represent at half the sample rate; the frame count (and window
size) set the spectrogram's width and height — the 'image' a downstream CNN or transformer sees.

</details>